# Simulating Data From Bayesian Networks

pgmpy implements the `DiscreteBayesianNetwork.simulate` method to allow users to simulate data from a fully defined Bayesian Network under various conditions. These conditions can be any combination of:
1. Virtual Evidence
2. Hard Evidence
3. Virtual Intervention
4. Hard Intervention

Users can also provide data corresponding to some of the variables in the model (partial samples) to the simulation method. This allows users to fix the values of those variables to the specified value.

Lastly, the user can also generate data with missing values, according to a user-defined CPD, to simulate realistic real-world data and evaluate how missingness affects inference.

In [1]:
# A helper function to compute probability distributions from simulated samples.
def get_distribution(samples, variables=None):
    """
    For marginal distribution, P(A): get_distribution(samples, variables=['A'])
    For joint distribution, P(A, B): get_distribution(samples, variables=['A', 'B'])
    """
    if variables is None:
        raise ValueError("variables must be specified")

    return samples.groupby(variables, observed=False).size() / samples.shape[0]

In [2]:
# Do not print warnings
import logging
from pgmpy.global_vars import logger
logger.setLevel(logging.ERROR)

# Specify the model to simulate data from.
from pgmpy.factors.discrete import TabularCPD
from pgmpy.example_models import load_model

import numpy as np
import pandas as pd

alarm = load_model("bnlearn/alarm")

c:\Users\Anusa\anaconda3\envs\pgmpy\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Standard simulation

Without any specified conditions for simulation, the `simulate` method draws samples from the joint distribution of the model.

In [3]:
samples = alarm.simulate(n_samples=int(1e4))
samples.head()

D:\pgmpy\pgmpy\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from pgmpy.estimators.StructureScore import (
Generating for node: BP: 100%|██████████| 37/37 [00:00<00:00, 86.21it/s]         


,PRESS,MINVOLSET,CATECHOL,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,SAO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,LOW,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
1,NORMAL,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,NORMAL,FALSE,NORMAL,NORMAL,NORMAL
2,HIGH,NORMAL,HIGH,ZERO,LOW,NORMAL,NORMAL,ZERO,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
3,HIGH,NORMAL,HIGH,HIGH,LOW,NORMAL,HIGH,HIGH,LOW,HIGH,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,ZERO
4,HIGH,NORMAL,HIGH,LOW,NORMAL,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,NORMAL,FALSE,NORMAL,HIGH,FALSE,HIGH,LOW,NORMAL


## 2. Simulation under specified evidence

Specifying hard evidence for some variables fixes their values to the specified evidence value during simulation.

In [4]:
evidence = {"CVP": "NORMAL", "HR": "HIGH"}
samples = alarm.simulate(n_samples=int(1e4), evidence=evidence)
samples.head()

100%|██████████| 10000/10000 [00:00<00:00, 24649.44it/s]


,PRESS,MINVOLSET,CATECHOL,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,SAO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,HIGH,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
1,HIGH,NORMAL,HIGH,HIGH,LOW,NORMAL,HIGH,HIGH,LOW,NORMAL,...,TRUE,LOW,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
2,HIGH,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,LOW,HIGH,FALSE,HIGH,NORMAL,NORMAL
3,NORMAL,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
4,HIGH,NORMAL,NORMAL,ZERO,LOW,HIGH,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,LOW,HIGH,FALSE,HIGH,NORMAL,NORMAL


In [5]:
# All values of HR and CVP should be set to HIGH and NORMAL respectively.
print(all(samples.HR == "HIGH"))
print(all(samples.CVP == "NORMAL"))

True
True


## 3. Simulation under soft/virtual evidence

Unlike hard evidence where the value of the specified variables is fixed to the specified evidence, virtual evidence allows users to set the marginal distribution of variables.

In [6]:
# The virtual evidence is specified using TabularCPDs. Here, P(CVP=NORMAL) = 0.2, P(CVP=LOW) = 0.3, and P(CPV=HIGH) = 0.5
cvp_evidence = TabularCPD(variable="CVP",
                          variable_card=3,
                          values=[[0.2], [0.3], [0.5]],
                          state_names={"CVP": ["LOW", "NORMAL", "HIGH"]})
samples = alarm.simulate(n_samples=int(1e4), virtual_evidence=[cvp_evidence])

100%|██████████| 10000/10000 [00:00<00:00, 12764.23it/s]


In [7]:
# Check the marginal distribution of CVP
get_distribution(samples, variables=['CVP'])

CVP
HIGH      0.2383
LOW       0.0745
NORMAL    0.6872
dtype: float64

## 4. Simulation under specified intervention

Using the `do` argument, users can specify interventions to the model. The value of the intervened variables are set to the specified value and all incoming edges to these variables are removed in the model.

In [8]:
samples = alarm.simulate(n_samples=int(1e4), do={"CVP": "NORMAL", "HR": "HIGH"})
samples.head()

100%|██████████| 10000/10000 [00:01<00:00, 6792.30it/s]


,PRESS,MINVOLSET,CATECHOL,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,SAO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,NORMAL,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
1,LOW,HIGH,HIGH,HIGH,LOW,NORMAL,NORMAL,HIGH,LOW,HIGH,...,FALSE,NORMAL,NORMAL,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,HIGH
2,NORMAL,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
3,HIGH,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL
4,LOW,NORMAL,HIGH,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,LOW,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,NORMAL


## 5. Simulation under soft/virtual intervention

Similar to virtual evidence, users can specify virtual intervention as well.

In [9]:
cvp_intervention = TabularCPD(variable="CVP",
                              variable_card=3,
                              values=[[0.2], [0.3], [0.5]],
                              state_names={"CVP": ["LOW", "NORMAL", "HIGH"]})
samples = alarm.simulate(n_samples=int(1e4), virtual_intervention=[cvp_intervention])
get_distribution(samples, variables=["CVP"])  # P(HISTORY)

  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:00<00:00, 12488.74it/s]


CVP
HIGH      0.3798
LOW       0.2112
NORMAL    0.4090
dtype: float64

## 6. Partial samples

Users can also pass already generated data for some variables (for example, from some other simulation) to the simulation. This is equivalent to separately specifying evidence for each sample that is generate.

In [10]:
# Generate some data on CVP.
partial_cvp = pd.DataFrame(np.random.choice(["LOW", "NORMAL", "HIGH"], int(1e4)), columns=['CVP'])
samples = alarm.simulate(n_samples=int(1e4), partial_samples=partial_cvp)

Generating for node: BP: 100%|██████████| 37/37 [00:00<00:00, 126.94it/s]      


In [11]:
print(all(samples["CVP"] == partial_cvp["CVP"]))

True


## 7. Simulate missing data

Lastly, users can generate data with missing values for some specified variables, according to a user defined CPD. The name of the missing variable should be followed by a * to indicate missingness, and should contain 2 states: 1 (Missing) and 0 (Not Missing). Optionally, we can use the `return_full` argument to get back the removed values for comparison.

#### 7.1. Missing completely at random (MCAR)

In [12]:
# CVP data missing completely randomly with 0.4 probability
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.6], 
            [0.4]], # Missing probability = 0.4
    state_names={"CVP*": [0, 1]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 223.73it/s]      


,PRESS,MINVOLSET,CATECHOL,CVP_full,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,NORMAL,NORMAL,NORMAL,LOW,NORMAL,NORMAL,LOW,NORMAL,LOW,NORMAL,...,FALSE,NORMAL,NORMAL,TRUE,NORMAL,LOW,FALSE,LOW,LOW,NORMAL
1,NORMAL,NORMAL,HIGH,NORMAL,HIGH,LOW,NORMAL,HIGH,HIGH,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NORMAL,ZERO
2,HIGH,NORMAL,HIGH,LOW,ZERO,LOW,HIGH,HIGH,ZERO,HIGH,...,FALSE,NORMAL,NORMAL,TRUE,NORMAL,HIGH,FALSE,HIGH,LOW,NORMAL
3,ZERO,NORMAL,HIGH,LOW,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,TRUE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,LOW,NORMAL
4,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,NORMAL,FALSE,NORMAL,NaN,NORMAL


In [13]:
print(f"Missing values: {samples['CVP'].isna().sum()}/{len(samples['CVP'])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables="CVP_full"))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables="CVP_full")) # Since removal was completely random, we expect minimal change in distribution

Missing values: 384/1000

Original Distribution:
CVP_full
HIGH      0.148
LOW       0.122
NORMAL    0.730
dtype: float64

Distribution of Missing/Removed
CVP_full
HIGH      0.143229
LOW       0.109375
NORMAL    0.747396
dtype: float64


#### 7.2. Missing at random (MAR)

In [14]:
# CVP data missing depending on the observed LVEDVOLUME
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.8, 0.2, 0.7], 
            [0.2, 0.8, 0.3]], # Missing probabilities: LOW = 0.2, NORMAL = 0.8, HIGH = 0.3
    evidence=["LVEDVOLUME"],
    evidence_card=[3],
    state_names={
        "CVP*": [0, 1],
        "LVEDVOLUME": ["LOW", "NORMAL", "HIGH"]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: ERRCAUTER:   0%|          | 0/38 [00:00<?, ?it/s]   

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 250.79it/s]      


,PRESS,MINVOLSET,CATECHOL,CVP_full,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NaN,NORMAL
1,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,HIGH,FALSE,LOW,HIGH,FALSE,HIGH,NaN,NORMAL
2,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,NORMAL,FALSE,NORMAL,HIGH,FALSE,HIGH,NaN,NORMAL
3,HIGH,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NaN,NORMAL
4,NORMAL,NORMAL,HIGH,LOW,ZERO,LOW,HIGH,HIGH,ZERO,HIGH,...,FALSE,NORMAL,LOW,TRUE,NORMAL,HIGH,FALSE,HIGH,LOW,NORMAL


In [15]:
print(f"Missing values: {samples['CVP'].isna().sum()}/{len(samples['CVP'])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables=["LVEDVOLUME", "CVP_full"]))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables=["LVEDVOLUME", "CVP_full"])) # Since probability of missing is higher when LVEDVOLUME is "NORMAL" we expect distribution to be higher values there, and lesser otherwise

Missing values: 675/1000

Original Distribution:
LVEDVOLUME  CVP_full
HIGH        HIGH        0.139
            LOW         0.001
            NORMAL      0.065
LOW         HIGH        0.000
            LOW         0.088
            NORMAL      0.002
NORMAL      HIGH        0.011
            LOW         0.029
            NORMAL      0.665
dtype: float64

Distribution of Missing/Removed
LVEDVOLUME  CVP_full
HIGH        HIGH        0.081481
            LOW         0.000000
            NORMAL      0.029630
LOW         HIGH        0.000000
            LOW         0.025185
            NORMAL      0.000000
NORMAL      HIGH        0.014815
            LOW         0.032593
            NORMAL      0.816296
dtype: float64


#### 7.3 Missing not at random (MNAR)

In [16]:
# CVP data missing depending on the unobserved original CVP value
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.2, 0.4, 0.6], 
            [0.8, 0.6, 0.4]], # Missing probabilities: LOW = 0.8, NORMAL = 0.6, HIGH = 0.4
    evidence=["CVP"],
    evidence_card=[3],
    state_names={
        "CVP*": [0, 1],
        "CVP": ["LOW", "NORMAL", "HIGH"]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: HYPOVOLEMIA:   0%|          | 0/38 [00:00<?, ?it/s]

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 358.76it/s]


,PRESS,MINVOLSET,CATECHOL,CVP_full,VENTALV,EXPCO2,STROKEVOLUME,CO,MINVOL,ARTCO2,...,DISCONNECT,FIO2,BP,LVFAILURE,PAP,HREKG,ANAPHYLAXIS,HRSAT,CVP,VENTMACH
0,LOW,NORMAL,HIGH,NORMAL,ZERO,ZERO,NORMAL,NORMAL,ZERO,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,LOW,FALSE,LOW,NORMAL,NORMAL
1,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,HIGH,ZERO,HIGH,...,FALSE,NORMAL,LOW,FALSE,NORMAL,HIGH,FALSE,HIGH,NaN,NORMAL
2,NORMAL,NORMAL,HIGH,NORMAL,ZERO,LOW,NORMAL,NORMAL,ZERO,HIGH,...,FALSE,NORMAL,NORMAL,FALSE,NORMAL,LOW,FALSE,NORMAL,NORMAL,NORMAL
3,LOW,HIGH,HIGH,NORMAL,HIGH,LOW,NORMAL,HIGH,NORMAL,LOW,...,FALSE,NORMAL,LOW,FALSE,NORMAL,NORMAL,FALSE,NORMAL,NORMAL,HIGH
4,HIGH,NORMAL,HIGH,NORMAL,ZERO,LOW,HIGH,HIGH,ZERO,HIGH,...,FALSE,NORMAL,HIGH,FALSE,NORMAL,HIGH,FALSE,HIGH,NaN,NORMAL


In [17]:
print(f"Missing values: {samples['CVP'].isna().sum()}/{len(samples['CVP'])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables="CVP_full"))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables="CVP_full")) # Since probability of missing is higher when CVP is "LOW" and lower when "CVP" is high we expect missing distribution for "LOW" to be greater, and for "HIGH" to be lower

Missing values: 591/1000

Original Distribution:
CVP_full
HIGH      0.176
LOW       0.096
NORMAL    0.728
dtype: float64

Distribution of Missing/Removed
CVP_full
HIGH      0.116751
LOW       0.133672
NORMAL    0.749577
dtype: float64
